# Phase 9 — MLflow Experiment Tracking
Logs every model's hyperparameters and evaluation metrics into MLflow.
Run `mlflow ui` after this notebook to view the experiment comparison dashboard.

In [ ]:
import sys, time
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import mlflow
from pathlib import Path
from src.data import load_movies
from src.cf import UserUserCF, ItemItemCF, MatrixFactorization
from src.content import TFIDFRecommender, SentenceTransformerRecommender
from src.hybrid import HybridRecommender
from src.evaluate import evaluate_model
from src.ab_test import ABTestFramework

def mlflow_safe(metrics):
    """MLflow forbids '@' in metric names — replace with '_at_'."""
    return {k.replace('@', '_at_'): v for k, v in metrics.items()}

In [ ]:
train  = pd.read_csv('../data/train.csv')
val    = pd.read_csv('../data/val.csv')
movies = load_movies()
print(f'train: {train.shape}  val: {val.shape}')

## 1. MLflow Setup

In [ ]:
mlflow.set_tracking_uri('file:///' + str(Path('../mlflow_runs').resolve()))
mlflow.set_experiment('recsys-movielens-1m')
print(f'Tracking URI : {mlflow.get_tracking_uri()}')
print(f'Experiment   : recsys-movielens-1m')

## 2. Fit All Models

In [ ]:
print('Fitting models...')
t0   = time.time()
uucf = UserUserCF(K=50).fit(train)
iicf = ItemItemCF(K=50).fit(train)
print(f'  CF models: {time.time()-t0:.1f}s')

t0 = time.time()
mf = MatrixFactorization(n_factors=64, n_epochs=2, lr=0.005, reg=0.1, batch_size=2048)
mf.fit(train, val)
print(f'  MF:        {time.time()-t0:.1f}s')

t0    = time.time()
tfidf = TFIDFRecommender(max_features=500).fit(movies, train)
st    = SentenceTransformerRecommender().fit(movies, train)
print(f'  Content:   {time.time()-t0:.1f}s')

hybrid = HybridRecommender(iicf, st, alpha=1.0)
print('Done.')

## 3. Log Each Model to MLflow

In [ ]:
N_USERS = 200
K_LIST  = (5, 10)

model_configs = [
    (
        'user-user-cf',
        {'model': 'UserUserCF', 'K': 50},
        lambda uid: [m for m, _ in uucf.recommend(uid, n=10)]
    ),
    (
        'item-item-cf',
        {'model': 'ItemItemCF', 'K': 50},
        lambda uid: [m for m, _ in iicf.recommend(uid, n=10)]
    ),
    (
        'matrix-factorization',
        {'model': 'MatrixFactorization', 'n_factors': 64, 'n_epochs': 2,
         'lr': 0.005, 'reg': 0.1, 'batch_size': 2048},
        lambda uid: [m for m, _ in mf.recommend(uid, n=10)]
    ),
    (
        'tfidf-content',
        {'model': 'TFIDFRecommender', 'max_features': 500},
        lambda uid: [m for m, _ in tfidf.recommend(uid, train, n=10)]
    ),
    (
        'sentence-transformer',
        {'model': 'SentenceTransformerRecommender', 'backbone': 'all-MiniLM-L6-v2', 'embedding_dim': 384},
        lambda uid: [m for m, _ in st.recommend(uid, train, n=10)]
    ),
    (
        'hybrid-alpha-1.0',
        {'model': 'HybridRecommender', 'cf': 'ItemItemCF', 'content': 'SentenceTransformer', 'alpha': 1.0},
        lambda uid: [m for m, _ in hybrid.recommend(uid, train, n=10)]
    ),
]

all_results = {}
for run_name, params, rec_fn in model_configs:
    metrics = evaluate_model(rec_fn, val, k_list=K_LIST, n_users=N_USERS)
    all_results[run_name] = metrics

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_params({'n_val_users': N_USERS, 'k_eval': 10})
        mlflow.log_metrics(mlflow_safe(metrics))

    print(f'{run_name:<25}  NDCG@10={metrics["ndcg@10"]:.4f}  MAP={metrics["map"]:.4f}  logged.')

## 4. Log A/B Test Results

In [ ]:
ab = ABTestFramework(k=10)

ab_experiments = [
    ('ab-cf-vs-hybrid-0.6', HybridRecommender(iicf, st, alpha=0.6)),
    ('ab-cf-vs-hybrid-0.8', HybridRecommender(iicf, st, alpha=0.8)),
]

ctrl_fn = lambda uid: [m for m, _ in iicf.recommend(uid, n=10)]

for run_name, treatment_model in ab_experiments:
    treat_fn = lambda uid, tm=treatment_model: [m for m, _ in tm.recommend(uid, train, n=10)]
    result   = ab.run(ctrl_fn, treat_fn, val, n_users=N_USERS)

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            'test_type': 'paired_ttest',
            'control':   'ItemItemCF',
            'treatment': run_name,
            'metric':    'ndcg_at_10',
            'n_users':   result['n_users'],
        })
        mlflow.log_metrics({
            'control_ndcg_at_10':   result['control_mean'],
            'treatment_ndcg_at_10': result['treatment_mean'],
            'lift_pct':             result['lift_pct'],
            'p_value':              result['p_value'],
        })

    sig = '(significant)' if result['significant'] else '(not significant)'
    print(f"{run_name}  lift={result['lift_pct']:+.2f}%  p={result['p_value']}  {sig}  logged.")

## 5. Summary Table

In [ ]:
metrics_to_show = ['ndcg@5', 'ndcg@10', 'map', 'recall@10', 'precision@10']
rows = []
for name, res in all_results.items():
    rows.append({'run': name, **{m: round(res.get(m, 0), 4) for m in metrics_to_show}})

df = pd.DataFrame(rows).set_index('run')
print(df.to_string())
print()
print('All runs logged. View in MLflow UI:')
print('  cd recsys && mlflow ui --port 5000')
print('  then open http://localhost:5000')